In [13]:
import pandas as pd
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [14]:
df = pd.read_csv("Wilson_disease_dataset.csv",index_col=0)
df = df.reset_index(drop=True)

In [15]:
print(df.isnull().sum())

Age                                    0
Sex                                    0
Ceruloplasmin Level                 6025
Copper in Blood Serum               6131
Free Copper in Blood Serum          5933
Copper in Urine                     6033
ALT                                 6111
AST                                 5862
Total Bilirubin                     5880
Albumin                             5977
Alkaline Phosphatase (ALP)          6065
Prothrombin Time / INR              6017
Gamma-Glutamyl Transferase (GGT)    5935
Kayser-Fleischer Rings                 0
Neurological Symptoms Score         6119
Psychiatric Symptoms                   0
Cognitive Function Score            5948
Family History                         0
ATB7B Gene Mutation                    0
Region                                 0
Socioeconomic Status                   0
Alcohol Use                            0
BMI                                    0
Is_Wilson_Disease                      0
dtype: int64


In [16]:
missing_columns=['Ceruloplasmin Level','Copper in Blood Serum','Free Copper in Blood Serum','Copper in Urine','ALT','AST','Total Bilirubin','Albumin','Alkaline Phosphatase (ALP)', 'Prothrombin Time / INR','Gamma-Glutamyl Transferase (GGT)','Neurological Symptoms Score','Cognitive Function Score']
for col in missing_columns:
    df[col] = df[col].fillna(df[col].median())

In [17]:
notnorm_cols=['Sex','Kayser-Fleischer Rings','Psychiatric Symptoms','Family History','ATB7B Gene Mutation','Region','Socioeconomic Status','Alcohol Use','Is_Wilson_Disease']
def NormalizeData(df, notnorm_cols):
    df = df.copy()
    cols = [col for col in df.columns if col not in notnorm_cols]
    for col in cols:
        min_val = df[col].min()
        max_val = df[col].max()
        df[col] = (df[col] - min_val) / (max_val - min_val + (max_val == min_val))
    return df
normalized_df= NormalizeData(df, notnorm_cols)

In [18]:
label_encoders = {}
for col in normalized_df.select_dtypes(include=['object', 'bool']).columns:
    le = LabelEncoder()
    normalized_df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [19]:
def calculate_class_medians(train_df):
    class_stats = {}
    classes = train_df['Is_Wilson_Disease'].unique()
    for class_label in classes:
        class_data = train_df[train_df['Is_Wilson_Disease'] == class_label].drop(columns='Is_Wilson_Disease')
        
        medians = class_data.median()
        q75 = class_data.quantile(0.75)
        q25 = class_data.quantile(0.25)
        iqr = q75 - q25
        
        margins = iqr
        
        class_stats[class_label] = (medians, margins)
    return class_stats

In [20]:
def soft_set_membership(test_obj, class_stats, weights):
    memberships = {}
    for class_label, (means, margins) in class_stats.items():
        score = 0
        for i, feature_val in enumerate(test_obj):
            mean_val = means.iloc[i]
            margin = margins.iloc[i]
            if (mean_val - margin) <= feature_val <= (mean_val + margin):
                score += weights[i]
        memberships[class_label] = score
    return memberships

In [21]:
def evaluate_accuracy_with_weights(trainSet, testSet, weights):
    class_stats = calculate_class_medians(trainSet)
    correct = 0
    total = len(testSet)

    for i, row in enumerate(testSet.itertuples(index=False)):
        true_class = getattr(row, 'Is_Wilson_Disease')
        test_data = np.array(row[:-1])

        memberships = soft_set_membership(test_data, class_stats, weights)
        predicted_class = max(memberships, key=memberships.get)

        if predicted_class == true_class:
            correct += 1

    return correct / total

In [22]:
def normalize(weights):
    total = weights.sum()
    return weights / total

def roulette_wheel_selection(scored_population, num_selected):
    weights, fitnesses = zip(*scored_population)
    total_fitness = sum(fitnesses)
    
    if total_fitness == 0:
        return list(np.random.choice(weights, num_selected, replace=False))
    
    probabilities = [f / total_fitness for f in fitnesses]
    selected = []
    
    for _ in range(num_selected):
        r = np.random.rand()
        cumulative = 0
        for i, p in enumerate(probabilities):
            cumulative += p
            if r <= cumulative:
                selected.append(weights[i])
                break
    return selected

In [23]:
def genetic_algorithm_weight_optimization(trainSet, testSet, generations=10, pop_size=10, mutation_rate=0.1):
    n_features = trainSet.shape[1] - 1

    # Initialize population
    population = [np.random.dirichlet(np.ones(n_features)) for _ in range(pop_size)]
    best_weights = None
    best_accuracy = 0

    for t in range(generations):
        print(f"\nGeneration {t+1}/{generations}")
        scored_population = []

        total_accuracy = 0

        # Evaluate population
        for idx, weights in enumerate(population):
            acc = evaluate_accuracy_with_weights(trainSet, testSet, weights)
            scored_population.append((weights, acc))
            total_accuracy += acc

        # Average accuracy in the generation
        avg_accuracy = total_accuracy / pop_size
        print(f"  Average accuracy in generation: {avg_accuracy:.4f}")

        # Track the best individual
        scored_population.sort(key=lambda x: x[1], reverse=True)
        if scored_population[0][1] > best_accuracy:
            best_weights, best_accuracy = scored_population[0]
            print(f"New best accuracy found: {best_accuracy:.4f}")

        # Roulette wheel selection
        survivors = roulette_wheel_selection(scored_population, pop_size)

        # Create new population
        new_population = []
        while len(new_population) < pop_size:
            idx1, idx2 = np.random.choice(len(survivors), 2, replace=False)
            parent1, parent2 = survivors[idx1], survivors[idx2]

            eta = np.random.rand()
            child1 = parent1 + eta * (parent2 - parent1)
            child2 = parent2 + eta * (parent1 - parent2)

            for child in [child1, child2]:
                child = np.clip(child, 0, None)
                if child.sum() == 0:
                    child = np.random.dirichlet(np.ones(n_features))
                else:
                    child = child / child.sum()

                # Mutation
                if np.random.rand() < mutation_rate:
                    mutation = np.random.uniform(-0.05, 0.05, size=n_features)
                    child += mutation
                    child = np.clip(child, 0, None)
                    child = child / child.sum() if child.sum() > 0 else np.random.dirichlet(np.ones(n_features))

                new_population.append(child)

                if len(new_population) >= pop_size:
                    break

        population = new_population

    return best_weights, best_accuracy

In [24]:
reduced_df = normalized_df.sample(frac=0.3, random_state=42)

trainSet, testSet = train_test_split(reduced_df, test_size=0.3, random_state=42)

best_weights, best_accuracy = genetic_algorithm_weight_optimization(trainSet, testSet, generations=10, pop_size=10)

print("\nBest feature weights:")
feature_names = trainSet.drop(columns='Is_Wilson_Disease').columns
for name, w in zip(feature_names, best_weights):
    print(f"{name}: {w:.4f}")

print(f"\nBest accuracy: {best_accuracy * 100:.2f}%")


Generation 1/10
  Average accuracy in generation: 0.9587
New best accuracy found: 0.9981

Generation 2/10
  Average accuracy in generation: 0.9881
New best accuracy found: 0.9989

Generation 3/10
  Average accuracy in generation: 0.9858

Generation 4/10
  Average accuracy in generation: 0.9837

Generation 5/10
  Average accuracy in generation: 0.9869

Generation 6/10
  Average accuracy in generation: 0.9890

Generation 7/10
  Average accuracy in generation: 0.9884

Generation 8/10
  Average accuracy in generation: 0.9833

Generation 9/10
  Average accuracy in generation: 0.9840

Generation 10/10
  Average accuracy in generation: 0.9855

Best feature weights:
Age: 0.1112
Sex: 0.0365
Ceruloplasmin Level: 0.0381
Copper in Blood Serum: 0.0506
Free Copper in Blood Serum: 0.0728
Copper in Urine: 0.0383
ALT: 0.0245
AST: 0.0462
Total Bilirubin: 0.0164
Albumin: 0.0023
Alkaline Phosphatase (ALP): 0.0695
Prothrombin Time / INR: 0.0152
Gamma-Glutamyl Transferase (GGT): 0.0262
Kayser-Fleischer Rin